# TPU-Native "Pallas" FlashAttention (Kaggle TPU v3-8)

This notebook hand-writes a **FlashAttention** kernel in **JAX Pallas** — the
closest thing to "writing CUDA" for Google's TPUs — and runs it inside a tiny
LLaMA-style decoder.

Instead of letting the XLA compiler fuse a naive `softmax(Q Kᵀ) V` (which
materializes the full `[seq, seq]` score matrix in HBM), we:

1. **Tile** Q/K/V and map each block from **HBM → VMEM** explicitly via Pallas
   `BlockSpec` index maps.
2. Run the **online-softmax** recurrence so the score matrix is never fully
   materialized.
3. Skip causal blocks that lie entirely in the future.

> **Setup:** In the Kaggle sidebar set **Accelerator → TPU VM v3-8**, then run
> all cells top to bottom.

## 1. Setup JAX & confirm the TPU

In [ ]:
# On Kaggle TPU VMs JAX is usually preinstalled. If not, uncomment:
# !pip install -q "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
import jax, jax.numpy as jnp
print("JAX", jax.__version__)
print("Devices:", jax.devices())
assert any(d.platform == "tpu" for d in jax.devices()), "No TPU found — set Accelerator to TPU v3-8"


## 2. The Pallas FlashAttention kernel

The `index_map` lambdas below are the **manual HBM → VMEM mapping**: for each
grid point `(batch, head, q_block, kv_block)` they say exactly which block of the
big HBM array to stream into the chip's small VMEM scratchpad.

In [ ]:
import functools
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu

_NEG_INF = -1e30

def _flash_attention_kernel(q_ref, k_ref, v_ref, o_ref,
                            m_scratch, l_scratch, acc_scratch,
                            *, sm_scale, causal, block_q, block_k,
                            seq_len_q, seq_len_k):
    q_block_idx = pl.program_id(2)
    kv_block_idx = pl.program_id(3)
    num_kv_blocks = pl.num_programs(3)

    @pl.when(kv_block_idx == 0)
    def _init():
        m_scratch[...] = jnp.full_like(m_scratch, _NEG_INF)
        l_scratch[...] = jnp.zeros_like(l_scratch)
        acc_scratch[...] = jnp.zeros_like(acc_scratch)

    def _do_block():
        q = q_ref[0, 0].astype(jnp.float32)
        k = k_ref[0, 0].astype(jnp.float32)
        v = v_ref[0, 0].astype(jnp.float32)

        kv_idx = kv_block_idx * block_k + jax.lax.broadcasted_iota(jnp.int32, (block_k, 1), 0)
        kv_valid = kv_idx < seq_len_k
        k = jnp.where(kv_valid, k, 0.0)
        v = jnp.where(kv_valid, v, 0.0)

        s = jnp.dot(q, k.T, preferred_element_type=jnp.float32) * sm_scale
        q_pos = q_block_idx * block_q + jax.lax.broadcasted_iota(jnp.int32, (block_q, block_k), 0)
        k_pos = kv_block_idx * block_k + jax.lax.broadcasted_iota(jnp.int32, (block_q, block_k), 1)
        mask = k_pos < seq_len_k
        if causal:
            mask = jnp.logical_and(mask, q_pos >= k_pos)
        s = jnp.where(mask, s, _NEG_INF)

        m_prev = m_scratch[...]
        m_cur = jnp.max(s, axis=-1, keepdims=True)
        m_new = jnp.maximum(m_prev, m_cur)
        p = jnp.exp(s - m_new)
        alpha = jnp.exp(m_prev - m_new)
        l_scratch[...] = alpha * l_scratch[...] + jnp.sum(p, axis=-1, keepdims=True)
        acc_scratch[...] = acc_scratch[...] * alpha + jnp.dot(p, v, preferred_element_type=jnp.float32)
        m_scratch[...] = m_new

    if causal:
        last_q_pos = q_block_idx * block_q + (block_q - 1)
        first_k_pos = kv_block_idx * block_k
        pl.when(last_q_pos >= first_k_pos)(_do_block)
    else:
        _do_block()

    @pl.when(kv_block_idx == num_kv_blocks - 1)
    def _finalize():
        l = jnp.where(l_scratch[...] == 0.0, 1.0, l_scratch[...])
        o_ref[0, 0] = (acc_scratch[...] / l).astype(o_ref.dtype)


def flash_attention(q, k, v, *, causal=False, sm_scale=None,
                    block_q=128, block_k=128, interpret=False):
    batch, num_heads, seq_len_q, head_dim = q.shape
    num_kv_heads, seq_len_k = k.shape[1], k.shape[2]
    q_per_kv = num_heads // num_kv_heads   # 1 for MHA, >1 for grouped-query attn
    if sm_scale is None:
        sm_scale = 1.0 / (head_dim ** 0.5)
    grid = (batch, num_heads, pl.cdiv(seq_len_q, block_q), pl.cdiv(seq_len_k, block_k))
    q_spec = pl.BlockSpec((1, 1, block_q, head_dim), lambda b, h, i, j: (b, h, i, 0))
    # Each query head h reads KV head h // q_per_kv (grouped-query attention).
    k_spec = pl.BlockSpec((1, 1, block_k, head_dim), lambda b, h, i, j: (b, h // q_per_kv, j, 0))
    o_spec = pl.BlockSpec((1, 1, block_q, head_dim), lambda b, h, i, j: (b, h, i, 0))
    kernel = functools.partial(_flash_attention_kernel, sm_scale=sm_scale, causal=causal,
                               block_q=block_q, block_k=block_k,
                               seq_len_q=seq_len_q, seq_len_k=seq_len_k)
    return pl.pallas_call(
        kernel, grid=grid, in_specs=[q_spec, k_spec, k_spec], out_specs=o_spec,
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        scratch_shapes=[pltpu.VMEM((block_q, 1), jnp.float32),
                        pltpu.VMEM((block_q, 1), jnp.float32),
                        pltpu.VMEM((block_q, head_dim), jnp.float32)],
        compiler_params=pltpu.CompilerParams(
            dimension_semantics=("parallel", "parallel", "parallel", "arbitrary")),
        interpret=interpret, name="flash_attention_fwd")(q, k, v)


## 3. Correctness check vs. a naive reference attention

In [ ]:
def reference_attention(q, k, v, *, causal=False, sm_scale=None):
    head_dim = q.shape[-1]
    if sm_scale is None:
        sm_scale = 1.0 / (head_dim ** 0.5)
    scores = jnp.einsum("bhqd,bhkd->bhqk", q, k).astype(jnp.float32) * sm_scale
    if causal:
        sq, sk = q.shape[2], k.shape[2]
        qp = jax.lax.broadcasted_iota(jnp.int32, (sq, sk), 0)
        kp = jax.lax.broadcasted_iota(jnp.int32, (sq, sk), 1)
        scores = jnp.where(qp >= kp, scores, -1e30)
    w = jax.nn.softmax(scores, axis=-1)
    return jnp.einsum("bhqk,bhkd->bhqd", w, v).astype(q.dtype)

keys = jax.random.split(jax.random.PRNGKey(0), 3)
q = jax.random.normal(keys[0], (1, 4, 512, 128), jnp.float32)
k = jax.random.normal(keys[1], (1, 4, 512, 128), jnp.float32)
v = jax.random.normal(keys[2], (1, 4, 512, 128), jnp.float32)
for causal in (False, True):
    out = flash_attention(q, k, v, causal=causal)
    ref = reference_attention(q, k, v, causal=causal)
    print(f"causal={causal}: max abs error = {float(jnp.max(jnp.abs(out - ref))):.2e}")


## 4. Benchmark: custom Pallas kernel vs. default XLA attention

In [ ]:
import time

def bench(fn, *args, warmup=3, iters=20):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))
    t0 = time.perf_counter()
    for _ in range(iters):
        out = fn(*args)
    jax.block_until_ready(out)
    return (time.perf_counter() - t0) / iters

pallas_fn = jax.jit(lambda q, k, v: flash_attention(q, k, v, causal=True))
xla_fn = jax.jit(lambda q, k, v: reference_attention(q, k, v, causal=True))

print(f"{'seq_len':>8} | {'XLA (ms)':>10} | {'Pallas (ms)':>12} | {'speedup':>8}")
print("-" * 48)
for seq in [256, 512, 1024, 2048, 4096]:
    ks = jax.random.split(jax.random.PRNGKey(seq), 3)
    q = jax.random.normal(ks[0], (1, 8, seq, 128), jnp.float32)
    k = jax.random.normal(ks[1], (1, 8, seq, 128), jnp.float32)
    v = jax.random.normal(ks[2], (1, 8, seq, 128), jnp.float32)
    t_xla = bench(xla_fn, q, k, v)
    t_pallas = bench(pallas_fn, q, k, v)
    print(f"{seq:>8} | {t_xla*1e3:>10.3f} | {t_pallas*1e3:>12.3f} | {t_xla/t_pallas:>7.2f}x")


## 5. Tiny LLaMA decoder powered by the kernel

A small, randomly-initialized LLaMA (RMSNorm + RoPE + SwiGLU) whose causal
self-attention is our Pallas kernel. The model is untrained — the point is that
the **full inference path** flows through the custom kernel.

In [ ]:
import jax.numpy as jnp

DIM, N_LAYERS, N_HEADS, HEAD_DIM, FFN, VOCAB = 256, 4, 2, 128, 688, 256
EPS, THETA = 1e-5, 10000.0

def rms_norm(x, w):
    var = jnp.mean(jnp.square(x.astype(jnp.float32)), -1, keepdims=True)
    return (x.astype(jnp.float32) * jax.lax.rsqrt(var + EPS) * w).astype(x.dtype)

def rope_tables(seq, hd):
    half = hd // 2
    inv = 1.0 / (THETA ** (jnp.arange(0, half, dtype=jnp.float32) / half))
    f = jnp.outer(jnp.arange(seq, dtype=jnp.float32), inv)
    emb = jnp.concatenate([f, f], -1)
    return jnp.cos(emb), jnp.sin(emb)

def rotate_half(x):
    h = x.shape[-1] // 2
    return jnp.concatenate([-x[..., h:], x[..., :h]], -1)

def apply_rope(x, cos, sin):
    return x * cos[None, None] + rotate_half(x) * sin[None, None]

def init(key):
    ks = iter(jax.random.split(key, 4 + N_LAYERS * 7))
    nrm = lambda k, sh, s: jax.random.normal(k, sh, jnp.float32) * s
    ps = 1.0 / DIM ** 0.5
    p = {"embed": nrm(next(ks), (VOCAB, DIM), 0.02), "layers": []}
    for _ in range(N_LAYERS):
        p["layers"].append({
            "an": jnp.ones((DIM,)), "fn": jnp.ones((DIM,)),
            "wq": nrm(next(ks), (DIM, DIM), ps), "wk": nrm(next(ks), (DIM, DIM), ps),
            "wv": nrm(next(ks), (DIM, DIM), ps), "wo": nrm(next(ks), (DIM, DIM), ps),
            "wg": nrm(next(ks), (DIM, FFN), ps), "wu": nrm(next(ks), (DIM, FFN), ps),
            "wd": nrm(next(ks), (FFN, DIM), 1.0 / FFN ** 0.5)})
    p["final"] = jnp.ones((DIM,))
    p["head"] = nrm(next(ks), (DIM, VOCAB), ps)
    return p

def forward(p, tokens):
    b, seq = tokens.shape
    x = p["embed"][tokens]
    cos, sin = rope_tables(seq, HEAD_DIM)
    heads = lambda t: t.reshape(b, seq, N_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)
    for L in p["layers"]:
        h = rms_norm(x, L["an"])
        q = apply_rope(heads(h @ L["wq"]), cos, sin)
        k = apply_rope(heads(h @ L["wk"]), cos, sin)
        v = heads(h @ L["wv"])
        a = flash_attention(q, k, v, causal=True)
        a = a.transpose(0, 2, 1, 3).reshape(b, seq, DIM)
        x = x + a @ L["wo"]
        h = rms_norm(x, L["fn"])
        x = x + (jax.nn.silu(h @ L["wg"]) * (h @ L["wu"])) @ L["wd"]
    return rms_norm(x, p["final"]) @ p["head"]

def generate(p, prompt, n):
    toks = prompt
    for _ in range(n):
        nxt = jnp.argmax(forward(p, toks)[:, -1, :], -1, keepdims=True)
        toks = jnp.concatenate([toks, nxt], 1)
    return toks

params = init(jax.random.PRNGKey(0))
prompt = jnp.array([[72, 101, 108, 108, 111]])  # "Hello" bytes
out = generate(params, prompt, 16)
print("Prompt:", prompt.tolist()[0])
print("Output:", out.tolist()[0])


## 6. Grouped-query attention (GQA / MQA)

Modern LLaMA models share each group of query heads over a single key/value head,
shrinking the KV cache. Our kernel supports this for free: `k`/`v` simply carry
fewer heads and the K/V `BlockSpec` maps query head `h` to KV head
`h // (num_heads // num_kv_heads)`. Here 8 query heads share 2 KV heads.

In [ ]:
def repeat_kv(x, n):
    return jnp.repeat(x, n, axis=1)

ks = jax.random.split(jax.random.PRNGKey(3), 3)
qg = jax.random.normal(ks[0], (1, 8, 256, 128), jnp.float32)   # 8 query heads
kg = jax.random.normal(ks[1], (1, 2, 256, 128), jnp.float32)   # 2 KV heads
vg = jax.random.normal(ks[2], (1, 2, 256, 128), jnp.float32)
out_gqa = flash_attention(qg, kg, vg, causal=True)
ref_gqa = reference_attention(qg, repeat_kv(kg, 4), repeat_kv(vg, 4), causal=True)
print("GQA (8 q-heads / 2 kv-heads) max abs error:",
      float(jnp.max(jnp.abs(out_gqa - ref_gqa))))


## 7. KV-cache decoding

Generating token-by-token by re-running the whole prefix is O(T²). A **KV cache**
stores each layer's keys/values so a decode step only computes the *new* token and
attends it against the cache — reusing the same kernel with `causal=False`. It
produces the identical tokens to the naive loop.

In [ ]:
def prefill(params, tokens):
    b, seq = tokens.shape
    x = params["embed"][tokens]
    cos, sin = rope_tables(seq, HEAD_DIM)
    hd = lambda t, nh: t.reshape(b, seq, nh, HEAD_DIM).transpose(0, 2, 1, 3)
    cache = []
    for L in params["layers"]:
        h = rms_norm(x, L["an"])
        q = apply_rope(hd(h @ L["wq"], N_HEADS), cos, sin)
        k = apply_rope(hd(h @ L["wk"], N_HEADS), cos, sin)
        v = hd(h @ L["wv"], N_HEADS)
        a = flash_attention(q, k, v, causal=True).transpose(0, 2, 1, 3).reshape(b, seq, DIM)
        x = x + a @ L["wo"]
        cache.append((k, v))
        h = rms_norm(x, L["fn"])
        x = x + (jax.nn.silu(h @ L["wg"]) * (h @ L["wu"])) @ L["wd"]
    return (rms_norm(x, params["final"]) @ params["head"]), cache

def decode_step(params, token, cache, pos):
    b = token.shape[0]
    x = params["embed"][token]
    cos_all, sin_all = rope_tables(pos + 1, HEAD_DIM)
    cos, sin = cos_all[pos:pos + 1], sin_all[pos:pos + 1]
    hd = lambda t, nh: t.reshape(b, 1, nh, HEAD_DIM).transpose(0, 2, 1, 3)
    new_cache = []
    for L, (ck, cv) in zip(params["layers"], cache):
        h = rms_norm(x, L["an"])
        q = apply_rope(hd(h @ L["wq"], N_HEADS), cos, sin)
        k = apply_rope(hd(h @ L["wk"], N_HEADS), cos, sin)
        v = hd(h @ L["wv"], N_HEADS)
        k = jnp.concatenate([ck, k], axis=2); v = jnp.concatenate([cv, v], axis=2)
        a = flash_attention(q, k, v, causal=False).transpose(0, 2, 1, 3).reshape(b, 1, DIM)
        x = x + a @ L["wo"]
        new_cache.append((k, v))
        h = rms_norm(x, L["fn"])
        x = x + (jax.nn.silu(h @ L["wg"]) * (h @ L["wu"])) @ L["wd"]
    return (rms_norm(x, params["final"]) @ params["head"]), new_cache

def generate_cached(params, prompt, n):
    b, P = prompt.shape
    logits, cache = prefill(params, prompt)
    nxt = jnp.argmax(logits[:, -1, :], -1, keepdims=True)
    toks = jnp.concatenate([prompt, nxt], 1)
    for i in range(n - 1):
        logits, cache = decode_step(params, nxt, cache, P + i)
        nxt = jnp.argmax(logits[:, -1, :], -1, keepdims=True)
        toks = jnp.concatenate([toks, nxt], 1)
    return toks

prompt = jnp.array([[72, 101, 108, 108, 111]])
naive = generate(params, prompt, 16)
cached = generate_cached(params, prompt, 16)
print("naive :", naive.tolist()[0])
print("cached:", cached.tolist()[0])
print("identical:", bool(jnp.array_equal(naive, cached)))


## 8. Notes & extensions

- **Why it's faster:** the kernel keeps the `[block_q, block_k]` score tile in
  VMEM and streams K/V blocks, so HBM traffic and peak memory grow ~linearly in
  sequence length instead of quadratically — the gap widens as `seq_len` grows.
- **Manual DMA (advanced):** `BlockSpec` lets Pallas pipeline the HBM→VMEM
  copies for you. For fully manual control you can instead keep K/V in
  `memory_space=pltpu.ANY` and issue `pltpu.make_async_copy(src, dst, sem)`
  yourself inside the kernel — the same online-softmax math applies.
- **KV cache:** this demo recomputes the full prefix each decode step for
  clarity; a real deployment would cache K/V and run a single-query kernel.
- **Backward pass:** only the forward kernel is implemented (inference focus).